# 02. Очистка данных по шагам
Блокнот вызывает общие функции из `src/cleaning.py` по одной и показывает результат каждого шага. Тот же код запускается из командной строки: `python -m src.pipeline`. Исполняется целиком после перезапуска ядра.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import logging
import pandas as pd
from IPython.display import display, Markdown
from src import config as cfg, io_utils as io, cleaning as cl, validation
logging.basicConfig(level=logging.WARNING)
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 80)
MODE = 'debug'          # 'debug' - срез 200 тыс. строк, 'full' - весь набор
sfx = '_debug' if MODE == 'debug' else ''
SRC = cfg.DEBUG_SLICE if MODE == 'debug' else cfg.RAW_BRANDED
OUT_DIR = cfg.RESULTS / ('debug_nb' if MODE == 'debug' else 'full_nb')
stats = cl.Stats()

## Исходная таблица с явной схемой типов
Чтение с параметром `dtype` из `config.SCHEMA`; `keep_default_na=False` — заглушки вроде `n/a` не исчезают молча.

In [2]:
if MODE == 'debug' and not SRC.exists():
    io.make_slice()
df = io.read_csv_typed(SRC)
n0, c0 = df.shape
print(n0, c0, round(df.memory_usage(deep=True).sum() / 1e6, 1), 'МБ')
display(df.dtypes.astype(str).value_counts())

200000 21 72.1 МБ


category          13
string             3
datetime64[us]     2
int32              1
float32            1
datetime64[s]      1
Name: count, dtype: int64

## Шаг 1. Соединение с `food.csv` по `fdc_id`
Левое соединение 1:1 (ключ справочника проверен на уникальность). Контроль: число строк до и после и число строк без пары.

In [3]:
df = cl.join_food(df, stats)
display(stats.frame().tail(1))

,step,processed,changed,removed,note
0,join_food,200000,0,0,пропущено: нет food.csv


## Шаг 2. Имена столбцов и типы

In [4]:
df = cl.normalize_column_names(df, stats)
df = cl.cast_types(df, stats)
display(stats.frame().tail(2))

,step,processed,changed,removed,note
1,normalize_column_names,200000,0,0,переименовано столбцов: 0
2,cast_types,200000,0,0,типы приведены к config.SCHEMA


## Шаг 3. Скрытые пропуски
`n/a`, `none`, `nan`, пустые после обрезки пробелов, порция ≤ 0, даты до 1990 г. → `NA`.

In [5]:
before = df.isna().sum()
df = cl.replace_hidden_missing(df, stats)
display((df.isna().sum() - before).loc[lambda s: s > 0].to_frame('добавлено NA'))

,добавлено NA
ingredients,5
serving_size,4


## Шаг 4. Структура пропусков
Пропуски не удаляются и не заполняются модой или медианой: это выдумало бы данные. Проверяем, не сосредоточены ли они в одной группе (`data_source`).

In [6]:
df = cl.analyze_missing(df, stats, OUT_DIR)
display(pd.read_csv(OUT_DIR / 'missing_by_data_source.csv', index_col=0))

,ingredients,serving_size,brand_name,branded_food_category,household_serving_fulltext,package_weight
data_source,,,,,,
GDSN,0.042,0.000,99.992,84.338,12.220,99.992
LI,0.460,0.002,100.000,0.000,17.473,100.000


## Шаг 5. GTIN и повторы
Коды дополняются нулями до 14 знаков; полные дубликаты и повторы `fdc_id` удаляются; прежние версии одного GTIN помечаются флагом `is_latest_version`.

In [7]:
n = len(df)
df = cl.normalize_gtin(df, stats)
df = cl.drop_duplicates(df, stats)
print('строк было', n, 'стало', len(df))
display(df['is_latest_version'].value_counts())

строк было 200000 стало 200000


is_latest_version
True     190866
False      9134
Name: count, dtype: int64

## Шаг 6. Категории
Пробелы, регистр, словарь единиц (`GRM/GM→g`, `MLT→ml`, `MC→mcg`), редкие категории → группа `Other`.

In [8]:
df = cl.normalize_categories(df, stats)
display(df['serving_size_unit'].value_counts(dropna=False))
display(df['branded_food_category_grp'].value_counts().head(8))

serving_size_unit
g     173920
ml     26080
Name: count, dtype: int64

branded_food_category_grp
Popcorn, Peanuts, Seeds & Related Snacks    10068
Candy                                        9130
Cheese                                       8166
Cookies & Biscuits                           6082
Ice Cream & Frozen Yogurt                    5696
Pickles, Olives, Peppers & Relishes          5546
Chips, Pretzels & Snacks                     4909
Breads & Buns                                4723
Name: count, dtype: int64

## Шаг 7. Каналы продаж и текст
`trade_channel` — JSON-список (нарушение 1НФ) → `A|B` без повторов; из `ingredients` удаляется HTML, нормализуются пробелы.

In [9]:
df = cl.parse_trade_channel(df, stats)
df = cl.clean_text(df, stats)
display(df['trade_channel'].dropna().value_counts().head(5))

Series([], Name: count, dtype: int64)

## Шаг 8. Флаги пропусков и единая шкала
`serving_size_std` — порция в граммах или миллилитрах; `package_weight` разбирается в `package_weight_g` / `package_weight_ml`.

In [10]:
df = cl.add_missing_flags(df, stats)
df = cl.normalize_units(df, stats)
display(df[['serving_size', 'serving_size_unit', 'serving_size_std', 'serving_unit_std',
            'package_weight', 'package_weight_g', 'package_weight_ml']].dropna(subset=['package_weight']).head(6))

,serving_size,serving_size_unit,serving_size_std,serving_unit_std,package_weight,package_weight_g,package_weight_ml
199999,112.0,g,112.0,g,1.25 LBR,NaN,NaN


## Шаг 9. Предметные ограничения и выбросы
Даты не в будущем; `discontinued_date` не раньше `available_date`. Выбросы порции — двумя способами (IQR и MAD), без удаления: ставятся флаги.

In [11]:
df = cl.check_domain(df, stats)
df = cl.flag_outliers(df, stats, OUT_DIR)
import json; print(json.dumps(json.load(open(OUT_DIR / 'outliers.json')), indent=1)[:900])

{
 "rows": 200000,
 "iqr": 7251,
 "mad": 14406,
 "both": 6960,
 "either": 14697,
 "implausible": 0,
 "max_value": 2808.0,
 "bounds": {
  "g": {
   "q1": 28.0,
   "q3": 85.0,
   "iqr_low": -57.5,
   "iqr_high": 170.5,
   "median": 40.0,
   "mad": 18.75
  },
  "ml": {
   "q1": 30.0,
   "q3": 243.0,
   "iqr_low": -289.5,
   "iqr_high": 562.5,
   "median": 240.0,
   "mad": 115.0
  }
 }
}


## Итог: «было — стало»

In [12]:
df = cl.finalize(df, stats)
display(stats.frame())
print(f'Было {n0} x {c0}, стало {len(df)} x {df.shape[1]}; потеряно строк: {100 * (n0 - len(df)) / n0:.4f} %')

,step,processed,changed,removed,note
0,join_food,200000,0,0,пропущено: нет food.csv
1,normalize_column_names,200000,0,0,переименовано столбцов: 0
2,cast_types,200000,0,0,типы приведены к config.SCHEMA
3,replace_hidden_missing,200000,9,0,"ячеек заменено на NA: {""ingredients"": 5, ""serving_size<=0"": 4}"
4,analyze_missing,200000,0,0,пропуски не удаляются и не заполняются (см. отчёт)
5,normalize_gtin,200000,187910,0,некорректных кодов: 3
6,drop_duplicates,200000,9134,0,полных повторов: 0; повторов fdc_id: 0; прежних версий gtin: 9134 (flag)
7,normalize_categories,200000,3936,0,"{""brand_owner"": 3741, ""branded_food_category"": 126, ""household_serving_fullt..."
8,parse_trade_channel,200000,0,0,JSON-список -> 'A|B' без повторов
9,clean_text,200000,4610,0,"{""ingredients"": 4610, ""not_a_significant_source_of"": 0}"


Было 200000 x 21, стало 200000 x 36; потеряно строк: 0.0000 %


## Сохранение и сверка с командной строкой
Результат блокнота пишется в отдельный файл и сравнивается с результатом `python -m src.pipeline` построчно.

In [13]:
out = cfg.PROCESSED / f'clean_notebook{sfx}.parquet'
io.write_parquet(df, out)
cli = cfg.PROCESSED / f'clean{sfx}.parquet'
if cli.exists():
    validation.compare(cli, out)
else:
    print('Сначала выполните: python -m src.pipeline', '--mode debug' if MODE == 'debug' else '')

Совпадают построчно и по типам: 200000 строк, 36 столбцов


## Идемпотентность
Повторная очистка готовой таблицы ничего не меняет.

In [14]:
again = cl.clean_all(df.copy(), cl.Stats(), None)
pd.testing.assert_frame_equal(df, again, check_exact=True)
print('идемпотентность подтверждена')

идемпотентность подтверждена
